In [1]:
import kagglehub
import pandas as pd
import numpy as np
import ast

In [2]:
path = kagglehub.dataset_download(
    "rounakbanik/the-movies-dataset"
)
metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")
ratingsDF = pd.read_csv(f"{path}/ratings.csv")


C:\Users\Johnny\AppData\Local\Temp\ipykernel_17884\716476305.py:4: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")


# Dataset - The Movies Dataset by Rounak Banik
report by John Lopes & Taha Riyaan

## Description

"About Dataset
Context
These files contain metadata for all 45,000 movies listed in the Full MovieLens Dataset. The dataset consists of movies released on or before July 2017. Data points include cast, crew, plot keywords, budget, revenue, posters, release dates, languages, production companies, countries, TMDB vote counts and vote averages.

This dataset also has files containing 26 million ratings from 270,000 users for all 45,000 movies. Ratings are on a scale of 1-5 and have been obtained from the official GroupLens website.

Content
This dataset consists of the following files:

movies_metadata.csv: The main Movies Metadata file. Contains information on 45,000 movies featured in the Full MovieLens dataset. Features include posters, backdrops, budget, revenue, release dates, languages, production countries and companies.

keywords.csv: Contains the movie plot keywords for our MovieLens movies. Available in the form of a stringified JSON Object.

credits.csv: Consists of Cast and Crew Information for all our movies. Available in the form of a stringified JSON Object.

links.csv: The file that contains the TMDB and IMDB IDs of all the movies featured in the Full MovieLens dataset.

links_small.csv: Contains the TMDB and IMDB IDs of a small subset of 9,000 movies of the Full Dataset.

ratings_small.csv: The subset of 100,000 ratings from 700 users on 9,000 movies.

The Full MovieLens Dataset consisting of 26 million ratings and 750,000 tag applications from 270,000 users on all the 45,000 movies in this dataset can be accessed here

Acknowledgements
This dataset is an ensemble of data collected from TMDB and GroupLens.
The Movie Details, Credits and Keywords have been collected from the TMDB Open API. This product uses the TMDb API but is not endorsed or certified by TMDb. Their API also provides access to data on many additional movies, actors and actresses, crew members, and TV shows. You can try it for yourself here.

The Movie Links and Ratings have been obtained from the Official GroupLens website. The files are a part of the dataset available here



Inspiration
This dataset was assembled as part of my second Capstone Project for Springboard's Data Science Career Track. I wanted to perform an extensive EDA on Movie Data to narrate the history and the story of Cinema and use this metadata in combination with MovieLens ratings to build various types of Recommender Systems.

Both my notebooks are available as kernels with this dataset: The Story of Film and Movie Recommender Systems

Some of the things you can do with this dataset:
Predicting movie revenue and/or movie success based on a certain metric. What movies tend to get higher vote counts and vote averages on TMDB? Building Content Based and Collaborative Filtering Based Recommendation Engines."

We decided to go with this dataset because we enjoying watching movies

## Code

### Cleaning

### Study 1

subset = genres, revenue, runtime, production_companies, title

Jaccard

In [3]:
def extractGenreNames(genres):
    if pd.isna(genres):
        return set()
    if isinstance(genres, str):
        try:
            genres = ast.literal_eval(genres)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(genres, list):
        return {item['name'] for item in genres if isinstance(item, dict) and 'name' in item}
    return set()

def jaccardSimilarity(a, b):
    if not a and not b:
        return 1.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0

movie = "Minions"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieGenres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
print(movie + " genres: " + str(movieGenres))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    genres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'genres': genres,
        'jaccard_similarity': jaccardSimilarity(movieGenres, genres),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Minions genres: {'Comedy', 'Family', 'Adventure', 'Animation'}


,title,genres,jaccard_similarity
8610,Asterix the Gaul,"{Comedy, Family, Adventure, Animation}",1.0
8626,Asterix and Cleopatra,"{Comedy, Family, Adventure, Animation}",1.0
43156,Smurfs: The Lost Village,"{Adventure, Family, Comedy, Animation}",1.0
608,The Aristocats,"{Family, Comedy, Adventure, Animation}",1.0
41586,Tri bogatyrya i Shamakhanskaya tsaritsa,"{Family, Comedy, Adventure, Animation}",1.0
40608,Storks,"{Adventure, Family, Comedy, Animation}",1.0
39750,VeggieTales: Tomato Sawyer & Huckleberry Larry...,"{Family, Comedy, Adventure, Animation}",1.0
39734,VeggieTales: Dave and the Giant Pickle,"{Adventure, Family, Comedy, Animation}",1.0
43294,Cars 3,"{Family, Comedy, Adventure, Animation}",1.0
39334,Ice Age: Collision Course,"{Adventure, Family, Comedy, Animation}",1.0


In [4]:
def extractCompanyNames(companies):
    if pd.isna(companies):
        return set()
    if isinstance(companies, str):
        try:
            companies = ast.literal_eval(companies)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(companies, list):
        return {item['name'] for item in companies if isinstance(item, dict) and 'name' in item}
    return set()

movie = "Pocahontas"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieCompanies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
print(movie + " production companies: " + str(movieCompanies))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    companies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'companies': companies,
        'jaccard_similarity': jaccardSimilarity(movieCompanies, companies),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Pocahontas production companies: {'Walt Disney Pictures', 'Walt Disney Feature Animation'}


,title,companies,jaccard_similarity
1798,Mulan,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5744,Treasure Planet,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
10520,Chicken Little,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
359,The Lion King,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
4238,Atlantis: The Lost Empire,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5310,Lilo & Stitch,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3493,Dinosaur,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3891,The Emperor's New Groove,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
1980,The Rescuers Down Under,"{Walt Disney Pictures, Silver Screen Partners ...",0.666667
1520,Air Bud,{Walt Disney Pictures},0.500000


Manhattan

In [5]:
movie = "Mortal Kombat"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRevenue = movieRow.iloc[0]['revenue'] if not movieRow.empty else 0.0
print(movie + " revenue: " + str(movieRevenue))

manhattanDistances = []
for i, row in metaDataDF.iterrows():
    rev = row['revenue']
    if pd.isna(rev) or pd.isna(movieRevenue):
        dist = float('inf')
    else:
        dist = abs(rev - movieRevenue)
    manhattanDistances.append({
        'title': row['title'],
        'revenue': rev,
        'manhattan_distance': dist,
    })

manhattanDF = pd.DataFrame(manhattanDistances)
manhattanDF = manhattanDF[manhattanDF['title'] != movie]
manhattanDF = manhattanDF.sort_values('manhattan_distance')
manhattanDF.head(10)

Mortal Kombat revenue: 122195920.0


,title,revenue,manhattan_distance
1885,Poltergeist,122200000.0,4080.0
14494,Invictus,122233971.0,38051.0
21604,Prisoners,122126687.0,69233.0
7011,Peter Pan,121975011.0,220909.0
1631,MouseHunt,122417389.0,221469.0
489,Executive Decision,121969216.0,226704.0
14234,Surrogates,122444772.0,248852.0
10762,Nanny McPhee,122489822.0,293902.0
31322,Magic Mike XXL,122513057.0,317137.0
5255,Spirit: Stallion of the Cimarron,122563539.0,367619.0


Euclidean

In [6]:
movie = "Avatar"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRuntime = movieRow.iloc[0]['runtime'] if not movieRow.empty else 0.0
print(movie + " runtime: " + str(movieRuntime))

euclideanDistances = []
for i, row in metaDataDF.iterrows():
    rt = row['runtime']
    if pd.isna(rt) or pd.isna(movieRuntime):
        dist = float('inf')
    else:
        dist = np.sqrt((rt - movieRuntime) ** 2)
    euclideanDistances.append({
        'title': row['title'],
        'runtime': rt,
        'euclidean_distance': dist,
    })

euclideanDF = pd.DataFrame(euclideanDistances)
euclideanDF = euclideanDF[euclideanDF['title'] != movie]
euclideanDF = euclideanDF.sort_values('euclidean_distance')
euclideanDF.head(10)

Avatar runtime: 162.0


,title,runtime,euclidean_distance
38801,Toni Erdmann,162.0,0.0
14992,Bunty Aur Babli,162.0,0.0
18451,The Disappearance of Haruhi Suzumiya,162.0,0.0
36560,Garv: Pride and Honour,162.0,0.0
33610,Times of Joy and Sorrow,162.0,0.0
29622,Sarfarosh,162.0,0.0
33434,A Tale of Two Cities,162.0,0.0
12520,Marketa Lazarová,162.0,0.0
7934,How the West Was Won,162.0,0.0
36571,Rakht,162.0,0.0


Edit Distance

In [ ]:
def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previousRow = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        currentRow = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previousRow[j + 1] + 1
            deletions = currentRow[j] + 1
            substitutions = previousRow[j] + (c1 != c2)
            currentRow.append(min(insertions, deletions, substitutions))
        previousRow = currentRow
    return previousRow[-1]

movie = "Pacific Rum"
editDistances = []
for title in metaDataDF['title']:
    dist = levenshtein(movie, str(title))
    editDistances.append({
        'title': title,
        'edit_distance': dist,
    })

editDF = pd.DataFrame(editDistances).sort_values('edit_distance')
editDF.head(10)

,title,edit_distance
21123,Pacific Rim,1
5132,Panic Room,5
26192,Sacrifice,6
38348,Sacrifice,6
18385,Sacrifice,6
40995,Pacific Banana,6
30770,Traffic Jam,6
9239,Mimic 2,7
11792,Maniac Cop,7
35942,Panic,7


### Study 2

### Study 3

## References